## Plot GNSS Antenna Pattern

In [ ]:
%load_ext autoreload
%autoreload 2

# Read data from file
import os
import math
import pandas as pd
import pylupnt as pnt

# gps
df = pd.read_csv(pnt.find_file("gps_table.csv"))
gps_antenna_names = df.set_index("PRN")["LM_File"].to_dict()
# for gps that LM file is not available, use the ACE_file instead
for prn in gps_antenna_names.keys():
    if type(gps_antenna_names[prn]) == float:
        gps_antenna_names[prn] = df.loc[prn - 1, "ACE_File"]
print(gps_antenna_names)
gps_antennas = {k: pnt.Antenna(v) for k, v in gps_antenna_names.items()}

# galileo
galileo_antennas = {0: pnt.Antenna("Galileo_E1")}

# qzss
qzss_names = ["1R", "02", "03", "04", "05", "06", "07"]
qzss_antennas = {
    k: pnt.Antenna("QZSS_" + qzss_names[k - 1] + "_L1") for k in range(1, 5)
}

rx_antenna = pnt.Antenna("moongpsr")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_antenna_gain_patter_2D(
    ax: plt.Axes = None,
    antenna: pnt.Antenna = None,
    phi: np.ndarray = None,
    theta: np.ndarray = None,
    show_max: bool = True,
    name: str = None,
):
    if ax is not None:
        plt.sca(ax)
    if phi is None:
        phi = np.linspace(-180, 180, 500)  # [deg]
    if theta is None:
        theta = np.linspace(0, 90, 4)  # [deg]
    for az in theta:
        gain = antenna.compute_gain(az * pnt.RAD, phi * pnt.RAD)
        plt.plot(
            pnt.DEG * pnt.wrap_to_pi(phi * pnt.RAD),
            gain,
            label=f"$\\varphi = {az:.0f}^\\circ$",
        )
    if name is None:
        plt.title(rf"{antenna.name}")
    else:
        plt.title(rf"{name}")
    plt.xlabel("Boresite Angle $\\theta$ [deg]")
    plt.ylabel("Gain $G$ [dB]")
    plt.text(
        0.98,
        0.95,
        f"$G_{{\\max}} = {antenna.get_gain_matrix().max():.2f}$ dB",
        transform=plt.gca().transAxes,
        ha="right",
        va="top",
    )
    plt.grid()
    # Legend with titl
    if len(theta) > 1:
        plt.legend(title="Azimuth $\\varphi$")

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 4, figsize=(18, 4))

# gps II-R
plot_antenna_gain_patter_2D(
    antenna=gps_antennas[4], ax=axs[0], name="GPS SVN74 (Block III, L1)"
)  # III
plot_antenna_gain_patter_2D(
    antenna=galileo_antennas[0], ax=axs[1], name="GALILEO E1"
)  # Galileo
plot_antenna_gain_patter_2D(
    antenna=qzss_antennas[2], ax=axs[2], name="QZSS PRN 2 (L1)"
)  # QZSS 2
plot_antenna_gain_patter_2D(
    antenna=rx_antenna, ax=axs[3], name="Moon Receiver"
)  # Moon GPS-R
plt.xlim(-5, 5)
plt.ylim(10, 15)

savedir = pnt.get_output_dir() / "iono_delay" / "gnss_plots"
os.makedirs(savedir, exist_ok=True)
plt.tight_layout()
plt.savefig(savedir / "antenna_gain_pattern.pdf")
plt.legend()